# DengueRadar — Treino e avaliação temporal
Avaliação em **semanas posteriores** ao treinamento, com baseline sazonal simples e métricas fora da amostra. Um segundo modelo final é ajustado com todos os dados apenas **após** a avaliação e salvo para uso no aplicativo. Não confundir métricas de teste com erro de treinamento.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE = Path.cwd().parent if Path.cwd().name.lower() in ('notebooks', 'notebook') else Path.cwd()
DATA = BASE / 'data' / 'processed' / 'dataset_dengueradar_limpo.csv'
MODELS = BASE / 'models'
MODELS.mkdir(parents=True, exist_ok=True)
if not DATA.exists():
    raise FileNotFoundError(f'{DATA}: execute antes o notebook 01 com os CSVs reais.')
df = pd.read_csv(DATA).sort_values(['ano','semana_epidem']).reset_index(drop=True)
cols = ['ano','semana_epidem','temp_media','precipitacao_total','casos']
missing = set(cols) - set(df.columns)
if missing:
    raise ValueError(f'Colunas ausentes: {missing}')
if df[cols].isna().any().any() or df.duplicated(['ano','semana_epidem']).any():
    raise ValueError('Dados incompletos ou períodos duplicados. Corrija o notebook 01.')
if (df['casos'] < 0).any():
    raise ValueError('Casos negativos encontrados.')
FEATURES = ['semana_epidem','temp_media','precipitacao_total']
if len(df) < 40:
    raise ValueError('Menos de 40 semanas disponíveis: avaliação temporal não seria minimamente informativa.')
print(f'Dataset: {len(df)} semanas, {df.ano.min()} a {df.ano.max()}')
display(df.head())

## 1. Separação cronológica e baseline

In [ ]:
n_test = max(8, int(np.ceil(len(df) * 0.20)))
train, test = df.iloc[:-n_test].copy(), df.iloc[-n_test:].copy()
# Baseline: mediana dos casos de cada semana do ano no TREINO; fallback para mediana global.
ref_week = train.groupby('semana_epidem')['casos'].median()
ref_global = float(train['casos'].median())
base_pred = test['semana_epidem'].map(ref_week).fillna(ref_global).to_numpy()
print(f'Treino {len(train)} semanas | teste cronológico {len(test)} semanas')
print('Teste inicia em:', test.iloc[0][['ano','semana_epidem']].to_dict())

## 2. Treinamento, avaliação sem vazamento e comparação

In [ ]:
modelo_teste = RandomForestRegressor(n_estimators=250, min_samples_leaf=2, random_state=42, n_jobs=-1)
modelo_teste.fit(train[FEATURES],train['casos'])
pred = np.maximum(0, modelo_teste.predict(test[FEATURES]))

def metr(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    wape = float(100 * np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) if np.sum(np.abs(y_true)) > 0 else None
    r2 = float(r2_score(y_true, y_pred)) if len(y_true)>1 else None
    return {'MAE': round(mae,3), 'RMSE':round(rmse,3), 'WAPE_pct':round(wape,3) if wape is not None else None, 'R2':round(r2,3) if r2 is not None else None}

metricas_rf = metr(test['casos'], pred)
metricas_baseline = metr(test['casos'],base_pred)
print('Random Forest (teste futuro):', metricas_rf)
print('Baseline sazonal (teste futuro):', metricas_baseline)
if metricas_rf['MAE'] >= metricas_baseline['MAE']:
    print('ALERTA: modelo não superou baseline sazonal em MAE no teste. Não afirmar superioridade preditiva.')
previsoes_teste = test[['ano','semana_epidem','casos']].copy()
previsoes_teste['previsto_rf'] = pred
previsoes_teste['previsto_baseline'] = base_pred
previsoes_teste.to_csv(MODELS/'avaliacao_temporal.csv', index=False)
display(previsoes_teste.tail())

## 3. Artefatos finais e metadados de avaliação

In [ ]:
# Ajuste de implantação com o histórico completo. As métricas continuam sendo as do teste anterior.
modelo_final = RandomForestRegressor(n_estimators=250, min_samples_leaf=2, random_state=42, n_jobs=-1)
modelo_final.fit(df[FEATURES],df['casos'])
joblib.dump(modelo_final, MODELS/'modelo_rf_dengue.pkl')
joblib.dump(FEATURES, MODELS/'features_dengue.pkl')
metadata = {
    'modelo': 'RandomForestRegressor', 'validacao': 'holdout cronologico (ultimas semanas)',
    'features': FEATURES, 'registros_totais':int(len(df)), 'registros_treino':int(len(train)),
    'registros_teste':int(len(test)), 'teste_inicio':{'ano':int(test.iloc[0].ano),'semana':int(test.iloc[0].semana_epidem)},
    'teste_fim':{'ano':int(test.iloc[-1].ano),'semana':int(test.iloc[-1].semana_epidem)},
    'metricas_rf_teste':metricas_rf, 'metricas_baseline_teste':metricas_baseline,
    'observacao':'Métricas calculadas antes do ajuste final, com semanas futuras ao treino; modelo final refitado no histórico completo.'
}
(MODELS/'avaliacao_modelo.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('Salvos: modelo, features, metadados e previsões de teste em', MODELS)